# PROJET MARKETING - ANALYSE COHORTES, RFM & CLV
## Partie 1 : Notebook d'Exploration Visuelle Complète

**Dataset:** Online Retail II (UCI)  
**Période:** 01/12/2009 - 09/12/2011  
**Objectif:** Exploration exhaustive pour cadrer l'application Streamlit

In [ ]:
# Import des bibliothèques
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("="*80)
print("EXPLORATION VISUELLE - ONLINE RETAIL II")
print("Analyse pour Application Marketing : Cohortes, RFM & CLV")
print("="*80)

## 1. CHARGEMENT DES DONNÉES

Le dataset Online Retail II est disponible sur UCI ML Repository.  
Format attendu: fichier Excel (.xlsx) avec colonnes:
- Invoice
- StockCode  
- Description
- Quantity
- InvoiceDate
- Price
- Customer ID
- Country

In [ ]:
# Chargement des données
# Option 1: Si vous avez téléchargé le fichier Excel
try:
    # Essayer de charger Year 2009-2010
    df1 = pd.read_excel('online_retail_II.xlsx', sheet_name='Year 2009-2010')
    # Essayer de charger Year 2010-2011  
    df2 = pd.read_excel('online_retail_II.xlsx', sheet_name='Year 2010-2011')
    df = pd.concat([df1, df2], ignore_index=True)
    print("✓ Données chargées depuis fichier Excel local")
except:
    print("⚠ Fichier Excel non trouvé")
    print("\nOptions de chargement:")
    print("1. Télécharger depuis: https://archive.ics.uci.edu/dataset/502/online+retail+ii")
    print("2. Placer le fichier 'online_retail_II.xlsx' dans le répertoire courant")
    print("3. Utiliser pd.read_excel() avec le chemin approprié")
    
print(f"\nDimensions du dataset: {df.shape}")
print(f"Nombre de lignes: {len(df):,}")
print(f"Nombre de colonnes: {df.shape[1]}")

## 2. FICHE SYNTHÉTIQUE DES DONNÉES

In [ ]:
print("="*80)
print("FICHE SYNTHÉTIQUE")
print("="*80)

print(f"""
SOURCE:
  - Nom: Online Retail II
  - Origine: UCI Machine Learning Repository
  - URL: https://archive.ics.uci.edu/dataset/502/online+retail+ii
  
DESCRIPTION:
  - Type: Données transactionnelles e-commerce
  - Secteur: Détaillant UK (vente en ligne et grossiste)
  - Nature: Ventes d'articles de cadeaux et décorations

COUVERTURE TEMPORELLE:
  - Période: 01/12/2009 - 09/12/2011
  - Durée: ~24 mois
  
VOLUME:
  - Nombre de lignes: {len(df):,}
  - Taille mémoire: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB
  - Clients uniques: {df['Customer ID'].nunique():,}
  - Factures uniques: {df['Invoice'].nunique():,}
  - Produits uniques: {df['StockCode'].nunique():,}
  - Pays: {df['Country'].nunique()}
""")

## 3. DICTIONNAIRE DES VARIABLES

Documentation complète de chaque variable avec type, sémantique et unités.

In [ ]:
print("="*80)
print("DICTIONNAIRE DES VARIABLES")
print("="*80)

variables_dict = {
    'Invoice': {
        'Type': 'object (string)',
        'Sémantique': 'Numéro de facture unique identifiant une transaction',
        'Unités/Valeurs': 'Chaîne alphanumérique. Commence par "C" pour annulations/retours',
        'Exemple': '536365, C536365'
    },
    'StockCode': {
        'Type': 'object (string)',
        'Sémantique': 'Code article/produit unique',
        'Unités/Valeurs': 'Chaîne alphanumérique',
        'Exemple': '85123A, POST'
    },
    'Description': {
        'Type': 'object (string)',
        'Sémantique': 'Libellé descriptif du produit',
        'Unités/Valeurs': 'Texte libre',
        'Exemple': 'WHITE HANGING HEART T-LIGHT HOLDER'
    },
    'Quantity': {
        'Type': 'int64',
        'Sémantique': 'Quantité commandée par ligne de facture',
        'Unités/Valeurs': 'Entier (négatif = retour/annulation)',
        'Exemple': '6, -2'
    },
    'InvoiceDate': {
        'Type': 'datetime64',
        'Sémantique': 'Date et heure de la transaction',
        'Unités/Valeurs': 'Format: YYYY-MM-DD HH:MM:SS',
        'Exemple': '2010-12-01 08:26:00'
    },
    'Price': {
        'Type': 'float64',
        'Sémantique': 'Prix unitaire hors taxes',
        'Unités/Valeurs': 'GBP (Livre Sterling £)',
        'Exemple': '2.55, 3.39'
    },
    'Customer ID': {
        'Type': 'float64',
        'Sémantique': 'Identifiant client anonymisé unique',
        'Unités/Valeurs': 'Entier (peut être NaN pour invités)',
        'Exemple': '17850, 13047'
    },
    'Country': {
        'Type': 'object (string)',
        'Sémantique': 'Pays de destination de la commande',
        'Unités/Valeurs': 'Nom de pays en anglais',
        'Exemple': 'United Kingdom, France, Germany'
    }
}

# Affichage formaté
for var, info in variables_dict.items():
    print(f"\n{var}")
    print("-" * 60)
    for key, value in info.items():
        print(f"  {key:<20}: {value}")

In [ ]:
# Aperçu des données
print("\n" + "="*80)
print("APERÇU DES DONNÉES")
print("="*80)
df.head(10)

In [ ]:
# Informations détaillées
print("\n" + "="*80)
print("INFORMATIONS DÉTAILLÉES")
print("="*80)
df.info()

In [ ]:
# Statistiques descriptives
print("\n" + "="*80)
print("STATISTIQUES DESCRIPTIVES")
print("="*80)
df.describe(include='all')

## 4. QUALITÉ DES DONNÉES

Analyse de la qualité: valeurs manquantes, doublons, outliers, règles métier.

In [ ]:
print("="*80)
print("4. QUALITÉ DES DONNÉES")
print("="*80)

# 4.1 Valeurs manquantes
print("\n[4.1] VALEURS MANQUANTES")
print("-"*60)

missing_data = pd.DataFrame({
    'Nombre_manquant': df.isnull().sum(),
    'Pourcentage': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_data = missing_data[missing_data['Nombre_manquant'] > 0].sort_values(
    'Pourcentage', ascending=False
)

print(missing_data)

print("\nINTERPRÉTATION:")
if 'Customer ID' in missing_data.index:
    pct_missing_cust = missing_data.loc['Customer ID', 'Pourcentage']
    print(f"  - {pct_missing_cust:.1f}% des transactions sans Customer ID")
    print("  - Probablement des ventes invités (sans compte client)")
    print("  - À EXCLURE des analyses de rétention et CLV")
    
if 'Description' in missing_data.index:
    print("  - Descriptions manquantes: possibles erreurs saisie ou produits spéciaux")

In [ ]:
# 4.2 Doublons
print("\n[4.2] DOUBLONS")
print("-"*60)

duplicates = df.duplicated().sum()
print(f"Lignes dupliquées complètes: {duplicates:,}")
print(f"Pourcentage: {duplicates/len(df)*100:.2f}%")

if duplicates > 0:
    print("\nINTERPRÉTATION:")
    print("  - Possibles erreurs de saisie ou exports multiples")
    print("  - À investiguer avant nettoyage")
else:
    print("\n✓ Aucun doublon détecté")

In [ ]:
# 4.3 Règles d'annulation
print("\n[4.3] RÈGLES D'ANNULATION")
print("-"*60)

df['IsCancelled'] = df['Invoice'].astype(str).str.startswith('C')
cancelled_count = df['IsCancelled'].sum()
cancelled_pct = (cancelled_count / len(df)) * 100

print(f"Factures d'annulation (Invoice commence par 'C'): {cancelled_count:,}")
print(f"Pourcentage: {cancelled_pct:.2f}%")

print("\nINTERPRÉTATION:")
print("  - Transactions de retours/annulations")
print("  - Quantités généralement négatives")
print("  - Impact sur:")
print("    * Taux de retour effectif")
print("    * Marge nette")
print("    * Analyse de rétention (clients insatisfaits?)")
print("    * CLV ajustée")

In [ ]:
# 4.4 Outliers - Quantity
print("\n[4.4] OUTLIERS - QUANTITY")
print("-"*60)

print(df['Quantity'].describe())
print(f"\nQuantités négatives: {(df['Quantity'] < 0).sum():,} lignes")
print(f"Quantités = 0: {(df['Quantity'] == 0).sum():,} lignes")
print(f"Quantités > 1000: {(df['Quantity'] > 1000).sum():,} lignes")
print(f"Quantité maximale: {df['Quantity'].max():,}")

print("\nINTERPRÉTATION:")
print("  - Quantités négatives = retours (cohérent avec factures 'C')")
print("  - Grandes quantités = probablement ventes grossistes/B2B")
print("  - Quantités à 0 = anomalies à investiguer")

In [ ]:
# 4.4 Outliers - Price
print("\n[4.4] OUTLIERS - PRICE")
print("-"*60)

print(df['Price'].describe())
print(f"\nPrix = 0: {(df['Price'] == 0).sum():,} lignes")
print(f"Prix négatifs: {(df['Price'] < 0).sum():,} lignes")
print(f"Prix > 1000£: {(df['Price'] > 1000).sum():,} lignes")
print(f"Prix maximum: {df['Price'].max():.2f}£")

print("\nINTERPRÉTATION:")
print("  - Prix à 0 ou négatifs = anomalies ou échantillons gratuits")
print("  - Prix très élevés = produits premium ou lots wholesale")
print("  - À filtrer selon contexte d'analyse")

In [ ]:
# 4.5 Granularité temporelle
print("\n[4.5] GRANULARITÉ TEMPORELLE")
print("-"*60)

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

print(f"Date minimale: {df['InvoiceDate'].min()}")
print(f"Date maximale: {df['InvoiceDate'].max()}")
print(f"Période couverte: {(df['InvoiceDate'].max() - df['InvoiceDate'].min()).days} jours")

# Extraction composantes temporelles
df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')
df['Day'] = df['InvoiceDate'].dt.day
df['Hour'] = df['InvoiceDate'].dt.hour
df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek
df['DayName'] = df['InvoiceDate'].dt.day_name()

print(f"\nNombre de jours uniques: {df['InvoiceDate'].dt.date.nunique()}")
print(f"Nombre de mois uniques: {df['YearMonth'].nunique()}")
print(f"Nombre d'années: {df['Year'].nunique()}")

print("\nINTERPRÉTATION:")
print("  - Données quotidiennes sur ~2 ans")
print("  - Suffisant pour analyses cohortes mensuelles")
print("  - Permet détection saisonnalité et tendances")

## 5. PRÉPARATION DES DONNÉES POUR VISUALISATION

Nettoyage et création de variables dérivées pour les analyses.

In [ ]:
print("="*80)
print("5. PRÉPARATION DES DONNÉES")
print("="*80)

# Copie du dataframe
df_clean = df.copy()

# Calcul du montant total (TotalAmount = Quantity × Price)
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['Price']
print("✓ Colonne 'TotalAmount' créée (Quantity × Price)")

# Flag annulations déjà créé
print("✓ Flag 'IsCancelled' déjà créé")

# Dataset pour analyses principales
# Filtrage: Customer ID présent + Quantity > 0 + Price > 0
df_analysis = df_clean[
    (df_clean['Customer ID'].notna()) & 
    (df_clean['Quantity'] > 0) & 
    (df_clean['Price'] > 0)
].copy()

print(f"\nDataframe original: {len(df):,} lignes")
print(f"Après nettoyage: {len(df_analysis):,} lignes")
print(f"Lignes retirées: {len(df) - len(df_analysis):,} ({(len(df)-len(df_analysis))/len(df)*100:.1f}%)")

print("\nFILTRES APPLIQUÉS:")
print("  ✓ Customer ID non-null (pour analyses cohortes/CLV)")
print("  ✓ Quantity > 0 (exclusion retours pour analyses CA positif)")
print("  ✓ Price > 0 (exclusion anomalies/échantillons gratuits)")

print("\nNOTE: Le dataframe 'df_clean' conserve TOUTES les données")
print("      pour analyses spécifiques des retours.")

## 6. VISUALISATIONS EXPLORATOIRES

Série de 8 graphiques clés avec métriques définies et interprétations.

### Graphique 1: Distribution du Montant des Transactions

**MÉTRIQUE:** TotalAmount = Quantity × Price (£)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Histogramme (95% des données)
amounts_filtered = df_analysis[df_analysis['TotalAmount'] < df_analysis['TotalAmount'].quantile(0.95)]
axes[0].hist(amounts_filtered['TotalAmount'], bins=100, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].set_xlabel('Montant Total (£)', fontsize=11)
axes[0].set_ylabel('Fréquence', fontsize=11)
axes[0].set_title('Distribution des Montants de Transaction\n(95% des données)',
                  fontsize=12, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
axes[0].axvline(df_analysis['TotalAmount'].median(), color='red', linestyle='--',
                label=f"Médiane: {df_analysis['TotalAmount'].median():.2f}£")
axes[0].axvline(df_analysis['TotalAmount'].mean(), color='orange', linestyle='--',
                label=f"Moyenne: {df_analysis['TotalAmount'].mean():.2f}£")
axes[0].legend()

# Box plot
axes[1].boxplot(amounts_filtered['TotalAmount'], vert=True)
axes[1].set_ylabel('Montant Total (£)', fontsize=11)
axes[1].set_title('Box Plot - Montants de Transaction', fontsize=12, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("INTERPRÉTATION:")
print(f"  - Distribution très asymétrique (queue longue à droite)")
print(f"  - Médiane: {df_analysis['TotalAmount'].median():.2f}£")
print(f"  - Moyenne: {df_analysis['TotalAmount'].mean():.2f}£ (>médiane = asymétrie positive)")
print(f"  - Majorité des transactions < 100£")
print(f"  - Quelques transactions de très haute valeur (>1000£)")
print("\nIMPLICATIONS BUSINESS:")
print("  → Segmentation clients nécessaire (retail vs wholesale)")
print("  → Stratégies différenciées par tranche de valeur")
print("  → Potentiel d'augmentation panier moyen via upsell/cross-sell")

### Graphique 2: Évolution Temporelle des Ventes

**MÉTRIQUES:**
- CA Mensuel: Somme(TotalAmount) par mois
- Commandes: Nombre unique de factures
- Clients Actifs: Nombre unique de Customer ID

In [ ]:
# Agrégation mensuelle
monthly_sales = df_analysis.groupby('YearMonth').agg({
    'TotalAmount': 'sum',
    'Invoice': 'nunique',
    'Customer ID': 'nunique'
}).reset_index()

monthly_sales['YearMonth_str'] = monthly_sales['YearMonth'].astype(str)

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# CA mensuel
axes[0].plot(monthly_sales['YearMonth_str'], monthly_sales['TotalAmount'] / 1000,
             marker='o', linewidth=2, markersize=6, color='steelblue')
axes[0].set_xlabel('Mois', fontsize=11)
axes[0].set_ylabel('CA Mensuel (milliers £)', fontsize=11)
axes[0].set_title('TENDANCE: Chiffre d\'Affaires Mensuel',
                  fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Commandes et clients actifs
ax2_twin = axes[1].twinx()
axes[1].bar(monthly_sales['YearMonth_str'], monthly_sales['Invoice'],
            alpha=0.6, label='Commandes', color='lightcoral')
ax2_twin.plot(monthly_sales['YearMonth_str'], monthly_sales['Customer ID'],
              marker='s', linewidth=2, markersize=5, color='darkgreen',
              label='Clients actifs')

axes[1].set_xlabel('Mois', fontsize=11)
axes[1].set_ylabel('Nombre de Commandes', color='lightcoral', fontsize=11)
ax2_twin.set_ylabel('Nombre de Clients Actifs', color='darkgreen', fontsize=11)
axes[1].set_title('VOLUME: Commandes et Clients Actifs Mensuels',
                  fontsize=13, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].tick_params(axis='y', labelcolor='lightcoral')
ax2_twin.tick_params(axis='y', labelcolor='darkgreen')
axes[1].legend(loc='upper left')
ax2_twin.legend(loc='upper right')

plt.tight_layout()
plt.show()

print("INTERPRÉTATION - TENDANCE:")
print("  - Forte croissance visible en 2010-2011")
print("  - Pics d'activité en Q4 (octobre-novembre)")
print("  - Effet saisonnier marqué (fêtes de fin d'année)")
print("\nINTERPRÉTATION - VOLUME:")
print("  - Corrélation forte CA ↔ nombre de clients")
print("  - Croissance soutenue de la base clients")
print("  - Augmentation du panier moyen observable")
print("\nIMPLICATIONS BUSINESS:")
print("  → Planification stock/marketing selon saisonnalité")
print("  → Opportunité rétention post-pic Q4")
print("  → Acquisition critique: septembre-octobre")